# **Hevy workout analysis**
Exploring my lifting data pulled from the Hevy API (`scripts/hevy.py`).

### Load the data
Read the cleaned CSV into a pandas DataFrame, each row is one set

In [ ]:
import pandas as pd

df = pd.read_csv("../data/hevy.csv")
df.head()

### Most-logged exercises
Which exercises I've done the most sets of.

In [ ]:
df["exercise"].value_counts().head(10)

### Top set over time: main lifts
Interactive charts for 6 most weighted exercises, plotly for interactivity

In [ ]:
import plotly.express as px

lifts = df.dropna(subset=["weight_lbs"])
top_ex = lifts["exercise"].value_counts().head(6).index
lifts = lifts[lifts["exercise"].isin(top_ex)]

best = lifts.groupby(["date", "exercise"])["weight_lbs"].max().reset_index()

fig = px.line(
    best, x="date", y="weight_lbs",
    facet_col="exercise", facet_col_wrap=3,
    facet_row_spacing=0.18,   # space between rows
    facet_col_spacing=0.08,   # space between columns
    markers=True, template="plotly_white",
    title="Top set over time",
    height=800,
)
fig.update_yaxes(matches=None, showticklabels=True, title="")
fig.update_xaxes(title="")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[1], yshift=10))
fig.update_traces(line_color="#ff6b6b", marker_color="#ff6b6b")
fig.update_layout(margin=dict(t=120))   # space above everything
fig.show()

## before vs. now
First logged top set compared with most recent one, for every lift done 3+ times.
assisted lifts skipped

In [ ]:
# heaviest set for each exercise on each day, oldest first
sessions = (
    df.dropna(subset=["weight_lbs"])
      .groupby(["exercise", "date"])["weight_lbs"].max()
      .reset_index()
      .sort_values("date")
)

# only keep exercises done 3+ times
counts = sessions["exercise"].value_counts()
sessions = sessions[sessions["exercise"].isin(counts[counts >= 3].index)]

# skip assisted lifts (lower weight = better, so they'd look backwards)
sessions = sessions[~sessions["exercise"].str.contains("Assisted")]

# grab the first and last top set for each exercise
summary = sessions.groupby("exercise")["weight_lbs"].agg(before="first", now="last").reset_index()

# avoid dividing by zero
summary = summary[summary["before"] > 0]

# % change from before to now
summary["change"] = ((summary["now"] - summary["before"]) / summary["before"] * 100).round()

# label like "Leg Press (Machine)  (+177%)"
summary["label"] = summary["exercise"] + "  (+" + summary["change"].astype(int).astype(str) + "%)"

# sort so the biggest improvement ends up on top
summary = summary.sort_values("change")

# grouped horizontal bars: gray = before, coral = now
fig = px.bar(
    summary, y="label", x=["before", "now"],
    barmode="group", orientation="h",
    template="plotly_white",
    color_discrete_map={"before": "#d3d3d3", "now": "#ff6b6b"},
    title="Before vs. now — top set (lbs)",
    height=30 * len(summary) + 200,
)

# remove extra axis/legend titles
fig.update_layout(yaxis_title="", xaxis_title="lbs", legend_title="")

fig.show()

### Workout length over time
How long each session lasted, plus a rolling average to show the trend.

In [ ]:
# one row per workout (every set in a workout has the same duration, so take the first)
sessions = (
    df.groupby(["date", "workout"])["duration_min"]
      .first()
      .reset_index()
      .sort_values("date")
)

# rolling average of the last 10 workouts, to smooth out the noise
sessions["avg_10"] = sessions["duration_min"].rolling(10, min_periods=3).mean()

# each dot = one workout, colored by workout type
fig = px.scatter(
    sessions, x="date", y="duration_min",
    color="workout",
    template="plotly_white",
    title="Workout length over time",
    height=550,
)

# add the trend line on top
fig.add_scatter(
    x=sessions["date"], y=sessions["avg_10"],
    mode="lines", name="10-workout average",
    line=dict(color="#ff6b6b", width=3),
)

# clean axis labels
fig.update_layout(xaxis_title="", yaxis_title="minutes", legend_title="")
fig.show()